In [2]:
import torch
import torch.nn as nn

In [3]:
def drop_path(x:torch.Tensor,training,drop_prob:float=0.):
    if not training:
        return x
    keep_prob=1-drop_prob
    shape=(x.shape[0],)+(1,)*(x.ndim-1)
    random_tensor=torch.rand(shape,dtype=x.dtype,device=x.device)+keep_prob
    random_tensor.floor_()
    output=x.div(keep_prob)*random_tensor
    return output

class DropPath(nn.Module):
    """Drop paths (Stochastic Depth) per sample (when applied in main path of residual blocks).
    """
    def __init__(self, drop_prob:float=0.):
        super(DropPath, self).__init__()
        self.drop_prob = drop_prob

    def forward(self, x):
        return drop_path(x,self.training, self.drop_prob)

In [31]:
class MLP(nn.Module):
    def __init__(
        self,
        embd_dim:int=768,
        mlp_ratio:float=4.,
        drop_ratio:float=0.,
    ):
        super().__init__()
        self.fc1=nn.Linear(embd_dim,int(embd_dim*mlp_ratio))
        self.act=nn.GELU()
        self.fc2=nn.Linear(int(embd_dim*mlp_ratio),embd_dim)
        self.drop=nn.Dropout(drop_ratio)
    
    def forward(self,x:torch.Tensor):
        x=self.fc1(x)
        x=self.drop(x)
        x=self.act(x)
        x=self.fc2(x)
        x=self.drop(x)
        return x

class Attention(nn.Module):
    def __init__(
        self,
        embd_dim:int=768,
        head:int=8,
        attn_drop_ratio:float=0.,
        proj_drop_ratio:float=0.,
        mlp_ratio:float=4.,
        scale=None,
    ):
        super().__init__()
        self.head=head
        self.embd_dim=embd_dim
        self.head_dim=embd_dim//head
        self.qkv=nn.Linear(embd_dim,embd_dim*3)
        self.attn_drop=nn.Dropout(attn_drop_ratio)
        self.proj_drop=nn.Dropout(proj_drop_ratio)
        self.proj=MLP(embd_dim,mlp_ratio)
        self.scale=self.head_dim**-0.5 if not scale else scale
    
    def forward(self,x:torch.Tensor):
        #x.shape=[B,N,C]
        B,N,C=x.shape
        #[B,N,C]->[B,N,3C]->[B,N,3,C]->[3,B,N,C]->[3,B,N,H,D]->[3,B,H,N,D]
        qkv = self.qkv(x).reshape(B, N, 3, self.head, C // self.head).permute(2, 0, 3, 1, 4)
        q,k,v=torch.unbind(qkv,0)
        #[B,H,N,D]@[B,H,D,N]->[B,H,N,N]
        attn=q@k.transpose(-2,-1)*self.scale
        attn=attn.softmax(dim=-1)
        attn=self.attn_drop(attn)
        x=(attn@v).transpose(1,2).reshape(B,N,C)
        x=self.proj(x)
        x=self.proj_drop(x)
        return x

class Block(nn.Module):
    def __init__(
        self,
        embd_dim:int=768,
        head:int=8,
        attn_drop_ratio:float=0.,
        drop_path_ratio:float=0,
        mlp_drop_ratio:float=0.,
        mlp_ratio:float=4.,
    ):
        super().__init__()
        self.norm1=nn.LayerNorm(embd_dim)
        self.norm2=nn.LayerNorm(embd_dim)
        self.attention=Attention(embd_dim=embd_dim,head=head,attn_drop_ratio=attn_drop_ratio,proj_drop_ratio=mlp_drop_ratio
                                 ,mlp_ratio=mlp_ratio)
        
        self.drop_path=nn.Identity() if drop_path_ratio==0. else DropPath(drop_path_ratio)
        self.mlp=MLP(embd_dim,mlp_ratio,mlp_drop_ratio)

    def forward(self,x:torch.Tensor):
        x=x+self.drop_path(self.attention(self.norm1(x)))
        x=x+self.drop_path(self.mlp(self.norm2(x)))
        return x

In [25]:
class PatchEmbed(nn.Module):
    def __init__(self,img_size:int,path_size:int,embd_dim:int):
        super().__init__()
        self.img_size=img_size
        self.path_size=path_size
        self.embd_dim=embd_dim
        self.num_path=(img_size//path_size)**2
        self.proj=nn.Conv2d(3,embd_dim,kernel_size=path_size,stride=path_size)
        self.norm=nn.LayerNorm(embd_dim)
    
    def forward(self,x:torch.Tensor)->torch.Tensor:
        x=self.proj(x)
        x=x.flatten(2).transpose(1,2)
        x=self.norm(x)
        return x

In [26]:

def _init_weights(m):
    if isinstance(m, nn.Linear) or isinstance(m, nn.Conv2d):
        nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
        if m.bias is not None:
            nn.init.zeros_(m.bias)
    elif isinstance(m, nn.LayerNorm):
        nn.init.zeros_(m.bias)
        nn.init.ones_(m.weight)

In [27]:

class VIT(nn.Module):
    def __init__(
        self,
        img_size:int=224,
        path_size:int=16,
        num_classses:int=1000,
        embd_dim:int=768,
        head:int=8,
        mlp_ratio:float=4.,
        drop_path_ratio:float=0.2,
        attn_drop_ratio:float=0.,
        mlp_drop_ratio:float=0,
        distill:bool=False,
        representation_size:int=0,
        embd_layer=PatchEmbed,
        depth:int=12,
    ):
        super().__init__()
        self.img_size=img_size
        self.num_path=(img_size//path_size)**2
        self.token=2 if distill else 1
        self.embd_dim=embd_dim
        self.num_features=embd_dim
        self.patch_embd=embd_layer(img_size,path_size,embd_dim)
        self.norm=nn.LayerNorm(embd_dim)

        self.pos_emb=nn.Parameter(torch.randn(1,self.num_path+self.token,embd_dim))
        self.dist_token=nn.Parameter(torch.zeros(1,1,embd_dim)) if distill else None
        self.cls_token=nn.Parameter(torch.zeros(1,1,embd_dim))
        self.pos_drop=nn.Dropout(mlp_drop_ratio)
        
        dpr=[x.item() for x in torch.linspace(0,drop_path_ratio,depth)]
        
        self.blocks=nn.Sequential(
            *[
                Block(
                    embd_dim=embd_dim,
                    head=head,
                    attn_drop_ratio=attn_drop_ratio,
                    mlp_drop_ratio=mlp_drop_ratio,
                    drop_path_ratio=dpr[i],
                    mlp_ratio=mlp_ratio,
                )
                for i in range(depth)
            ]
        )

        if representation_size>0 and not distill:
            self.has_logits=True
            self.num_features=representation_size
            self.pre_logits=nn.Sequential(
                nn.Linear(embd_dim,representation_size),
                nn.GELU(),
            )
        else:
            self.has_logits=False
            self.pre_logits=nn.Identity()
        
        self.head=nn.Linear(self.num_features,num_classses)
        self.head_distill=None
        if distill:
            self.head_distill=nn.Linear(self.num_features,num_classses)
        
        nn.init.trunc_normal_(self.pos_emb,std=0.02)
        if self.dist_token:
            nn.init.trunc_normal_(self.dist_token,std=0.02)
        
        nn.init.trunc_normal_(self.cls_token,std=0.02)
        self.apply(_init_weights)
    
    def forward_features(self,x:torch.Tensor):
        # [B, C, H, W] -> [B, num_patches, embed_dim]
        x = self.patch_embd(x)  # [B, 196, 768]
        # [1, 1, 768] -> [B, 1, 768]
        cls_token = self.cls_token.expand(x.shape[0], -1, -1)
        if self.dist_token is None:
            x = torch.cat((cls_token, x), dim=1)  # [B, 197, 768]
        else:
            x = torch.cat((cls_token, self.dist_token.expand(x.shape[0], -1, -1), x), dim=1)

        x = self.pos_drop(x + self.pos_emb)
        x = self.blocks(x)
        x = self.norm(x)
        if self.dist_token is None:
            return self.pre_logits(x[:, 0])
        else:
            return x[:, 0], x[:, 1]
    
    def forward(self,x):
        x=self.forward_features(x)
        if self.head_distill is not None:
            x, x_dist = self.head(x[0]), self.head_distill(x[1])
            if self.training and not torch.jit.is_scripting():
                # during inference, return the average of both classifier predictions
                return x, x_dist
            else:
                return (x + x_dist) / 2
        else:
            x = self.head(x)
        return x

In [32]:
vit=VIT()
vit.cuda()
x=torch.randn(1,3,224,224)
x=x.cuda()
y=vit(x)
print(y.shape)

torch.Size([1, 1000])
